# Cross-Database CVE Verification with TensorFeed (Hosted MCP)

**The actual production failure mode for security agents is not hallucination. It's acting on a single source.**

When a finance agent reads a fabricated headline and trades on it, the model didn't hallucinate; it read the source faithfully and the source was wrong. The same shape applies to security: a triage agent that judges a CVE off one database can be wrong without ever fabricating anything. The fix is corroboration across independent sources.

This notebook builds a small agent that uses [TensorFeed.ai](https://tensorfeed.ai)'s hosted MCP server with the **OpenAI Responses API's native MCP tool integration** to compose three independent vulnerability databases (MITRE CVE List, CISA Known Exploited Vulnerabilities, FIRST.org EPSS) for any CVE, in a single Responses call.

**Why TensorFeed?** TF is a free hosted MCP server (`https://tensorfeed.ai/api/mcp`) that exposes 17 tools spanning AI news, model pricing, AI service status, security advisories, SEC EDGAR filings, FDA regulatory data, and US energy indicators. No auth required for the tools used here. License: most underlying data is US Government public domain or CC0; commercial redistribution permitted; attribution preserved on every response. TF is also published as a hosted server in the official Model Context Protocol Registry as `ai.tensorfeed/mcp-server`.

**Why OpenAI's Responses API for this?** The Responses API supports MCP tools natively (`tools[].type = "mcp"`). The model autonomously fans out tool calls, parses results, and composes a final answer in one Responses call. No manual loop required.

**What you'll see by the end:**
- A single Responses call that verifies one CVE across three security databases via the TF MCP server
- The `confirmed_by` corroboration pattern surfaced from the model's natural reasoning
- A second example doing parallel triage of three CVEs ranked by composite risk


## Prerequisites

- Python 3.11+
- An OpenAI API key in the `OPENAI_API_KEY` environment variable
- The `openai` package (>= the version that supports Responses API MCP tools)

```bash
pip install --upgrade openai
```

No TensorFeed account or API key needed. The three tools used in this notebook are part of TensorFeed's free tier.


In [ ]:
import os
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in your environment first.")

client = OpenAI(api_key=api_key)
MODEL = "gpt-5.1"  # any current chat-completions/responses-capable model


## Connect the TensorFeed MCP server as a tool

The Responses API takes the MCP server URL directly. The model decides which tools to call based on the user input + the descriptions the MCP server returns from `tools/list`. We constrain the allowed tools to the three security ones for this demo (the TF server exposes 17 in total; restricting keeps the agent focused on the verification task).


In [ ]:
TF_MCP_TOOL = {
    "type": "mcp",
    "server_label": "tensorfeed",
    "server_url": "https://tensorfeed.ai/api/mcp",
    # Restrict to the three security tools for this demo.
    # Omit allowed_tools to expose all 17 TF tools to the model.
    "allowed_tools": [
        "get_cve_record",
        "get_kev_catalog",
        "get_epss_score",
    ],
    "require_approval": "never",
}


## Demo 1: Verify a single CVE

CVE-2024-3094 (the XZ backdoor, March 2024) is a useful test case: it has a complete MITRE record and EPSS data; KEV presence varies by snapshot. The model should consult all three sources and report whichever subset has data.


In [ ]:
response = client.responses.create(
    model=MODEL,
    tools=[TF_MCP_TOOL],
    input=(
        "Verify CVE-2024-3094 across multiple databases. Call MITRE for the canonical record, "
        "check the CISA KEV catalog for active exploitation, and pull the FIRST.org EPSS score. "
        "Then summarize: severity_band, exploited_in_wild boolean (true if KEV has the CVE), "
        "epss_probability, a confirmed_by list of the databases that returned data, and a "
        "one-sentence triage recommendation."
    ),
)

print(response.output_text)


## Demo 2: Triage three CVEs by composite risk

A more realistic agent task: given a list of CVEs that landed in your security feed today, decide which to patch first. The model fans out across all three sources for each CVE and returns a ranked list with reasoning.


In [ ]:
triage = client.responses.create(
    model=MODEL,
    tools=[TF_MCP_TOOL],
    input="""I have three CVEs to triage. For each, look up MITRE/KEV/EPSS via tensorfeed
and rank them by patch priority. Brief reasoning per CVE plus an ordered list at the end.

CVEs to triage:
- CVE-2024-3094
- CVE-2023-44487
- CVE-2024-21626""",
)

print(triage.output_text)


## What just happened

The Responses API took our MCP server URL plus a natural-language instruction and:

1. Recognized "verify" implies cross-source corroboration
2. Sequenced the calls (MITRE first to confirm existence, KEV for exploitation, EPSS for likelihood)
3. Surfaced the `confirmed_by` list so the user can audit which databases backed the answer

For the triage task, the model fanned out across three CVEs (~9 tool calls total) and produced a ranked list. No hallucinated CVE ids, no made-up severity scores; everything is sourced. The whole thing is a single `responses.create` call from the developer's perspective.

## Going further: TensorFeed's premium one-call composition

TensorFeed also exposes a single premium endpoint that does this composition server-side: `/api/premium/security/verified/{cve_id}`. It joins MITRE + KEV + EPSS + OSV.dev + CISA Vulnrichment into one fact card with `confirmed_by` and `corroboration_count` fields, returning ~6,000 saved tokens per call vs the multi-tool approach. The premium endpoint requires a bearer token purchased via x402 V2 on Base mainnet ($0.02/credit) at https://tensorfeed.ai/developers/agent-payments.

For most agent workloads the free three-tool composition shown above is fine; reach for the premium endpoint when verifying CVEs at scale (large vuln scan output, continuous monitoring, RAG indexing).

## Resources

- TensorFeed.ai: https://tensorfeed.ai
- Endpoint catalog: https://tensorfeed.ai/api/meta
- Agent-friendly entry doc: https://tensorfeed.ai/llms.txt
- TF in the official MCP Registry: https://registry.modelcontextprotocol.io/ (`ai.tensorfeed/mcp-server`)
- Source code (public): https://github.com/RipperMercs/tensorfeed
